In [1]:
library(tidyverse)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


In [2]:
library("dbplyr")
library("RPostgres")
library("toml")


Attaching package: ‘dbplyr’


The following objects are masked from ‘package:dplyr’:

    ident, sql, sql_escape_ident, sql_escape_string




In [3]:
cfg <- toml::read_toml("../config.toml")

In [4]:
u <- httr2::url_parse(cfg$database_url)

In [5]:
con <- dbConnect(
  RPostgres::Postgres(),
  host     = u$hostname,
  port     = if (is.null(u$port)) 5432L else as.integer(u$port),
  dbname   = sub("^/", "", u$path),
  user     = u$username,
  password = u$password,
  sslmode  = if (isTRUE(cfg$pg_tls)) "require" else "prefer")

In [6]:
readings <- tbl(con, "office_fishtank_readings")

In [7]:
readings

# A query:  ?? x 3
# Database: postgres  [postgres@192.168.18.23:5432/postgres]
   time                sensor            value
   <dttm>              <chr>             <dbl>
 1 2026-10-04 15:26:53 water_temperature  21.7
 2 2026-10-04 15:27:33 water_temperature  21.7
 3 2026-10-04 15:28:33 water_temperature  21.7
 4 2026-10-04 15:29:33 water_temperature  21.7
 5 2026-10-04 15:30:33 water_temperature  21.7
 6 2026-10-04 15:31:33 water_temperature  21.7
 7 2026-10-04 15:32:53 water_temperature  21.7
 8 2026-10-04 15:33:33 water_temperature  21.7
 9 2026-10-04 15:34:33 water_temperature  21.7
10 2026-10-04 15:35:33 water_temperature  21.6
# ℹ more rows

In [8]:
readings %>%
  filter(time >= local(Sys.time() - 2 * 3600)) %>%
  arrange(time) %>%
  collect()

time,sensor,value
<dttm>,<chr>,<dbl>
2026-10-09 09:58:41,water_temperature,21.937500
2026-10-09 10:01:47,tds_voltage,0.164000
2026-10-09 10:01:47,tds,59.379960
2026-10-09 10:03:42,ph_voltage,1.422000
2026-10-09 10:03:42,ph,7.701255
2026-10-09 10:13:41,water_temperature,22.062500
2026-10-09 10:28:41,water_temperature,22.125000
2026-10-09 10:43:41,water_temperature,22.187500
2026-10-09 10:58:41,water_temperature,22.250000


# Trigger a reading

In [ ]:
R -e 'library(ps); cat("ok\n")'

In [ ]:
m <- cfg$mqtt

In [ ]:
args <- c(
  "-h", m$host,
  "-p", m$port,
  "-i", paste0("r_trigger_", Sys.getpid()),
  "-t", "fishtank/command/measure",
  "-q", "1",
  "-m", "1",
  "-u", m$user,
  "-P", m$password)

In [ ]:
system2("/opt/homebrew/bin/mosquitto_pub",
    args,
    stdout = "", stderr = "")

Sys.time()